In [11]:
# =============================================================================
# Celula 0: Fonte de dados (SQL ou CSV em output/data)
# =============================================================================
import os
import sys
from pathlib import Path

# --- Bootstrap para importar modulos compartilhados (notebooks/shared) ---
def _ensure_shared_path():
    cwd = Path.cwd()
    if (cwd / 'shared').exists():
        sys.path.insert(0, str(cwd))
    elif (cwd / 'notebooks' / 'shared').exists():
        sys.path.insert(0, str(cwd / 'notebooks'))

_ensure_shared_path()

from shared.data_sources import select_raw_source

DATA_DIR = os.path.join('output', 'data')
METRICS_DIR = os.path.join('output', 'metrics')
os.makedirs(METRICS_DIR, exist_ok=True)

CONFIG_PATH = os.path.join(METRICS_DIR, 'data_source_config.json')

DATA_SOURCE, CSV_FILE = select_raw_source(
    data_dir=DATA_DIR,
    config_path=CONFIG_PATH,
    allow_sql=True,
)


In [12]:
# =============================================================================
# Celula 1: Configuracao e Imports
# =============================================================================
import os
import sys
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats as scipy_stats
from scipy.signal import savgol_filter
from scipy.stats import skew, kurtosis, kruskal
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import oracledb

warnings.filterwarnings('ignore')

# --- Bootstrap para importar modulos compartilhados (notebooks/shared) ---
def _ensure_shared_path():
    cwd = Path.cwd()
    if (cwd / 'shared').exists():
        sys.path.insert(0, str(cwd))
    elif (cwd / 'notebooks' / 'shared').exists():
        sys.path.insert(0, str(cwd / 'notebooks'))

_ensure_shared_path()

from shared.class_config import (
    CLASS_ORDER, COLOR_MAP, LEGACY_CLASS_ORDER, LEGACY_COLOR_MAP,
    derive_composite_label, get_active_classes, get_color_map,
    FILTER_LABELS_SQL,
)
from shared.data_sources import load_raw_from_oracle

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# --- CONFIGURACAO DO BANCO (Oracle XE) ---
ORACLE_USER     = os.environ.get('ORACLE_USER', 'dersao')
ORACLE_PASSWORD = os.environ.get('ORACLE_PASSWORD', '986960440')
ORACLE_HOST     = os.environ.get('ORACLE_HOST', 'localhost')
ORACLE_PORT     = os.environ.get('ORACLE_PORT', '1521')
ORACLE_SERVICE  = os.environ.get('ORACLE_SERVICE_NAME', 'xepdb1')

# DSN para oracledb direto (thin mode): host:port/service funciona aqui
ORACLE_DSN_STR  = f'{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE}'

# IMPORTANTE: SQLAlchemy trata "host:port/nome" como SID (legado).
# Para SERVICE_NAME obrigatoriamente usar "?service_name=..."
ORACLE_CONN_STR = (
    f'oracle+oracledb://{ORACLE_USER}:{ORACLE_PASSWORD}'
    f'@{ORACLE_HOST}:{ORACLE_PORT}/?service_name={ORACLE_SERVICE}'
)

# =============================================================================
# AUTO-DETECT: Ler parametros do sistema (control_state.json + banco Oracle)
# =============================================================================
CONTROL_STATE_PATH = os.path.join(
    '..', 'api', 'control_states', 'control_state_ESP32_MPU6050_ORACLE.json'
)
if not os.path.exists(CONTROL_STATE_PATH):
    CONTROL_STATE_PATH = os.path.join('..', 'api', 'control_state.json')

# 1. Ler control_state.json (fonte de verdade do servidor)
control_state = {}
if os.path.exists(CONTROL_STATE_PATH):
    with open(CONTROL_STATE_PATH, 'r') as f:
        control_state = json.load(f)
    print(f'[AUTO] control_state.json carregado: {CONTROL_STATE_PATH}')
    print(f'  Modo atual:      {control_state.get("mode", "N/A")}')
    print(f'  Sample rate:     {control_state.get("sample_rate", "N/A")} Hz')
    print(f'  Collection ID:   {control_state.get("collection_id", "N/A")}')
else:
    print(f'[AVISO] control_state.json nao encontrado em {CONTROL_STATE_PATH}')
    print(f'  Usando valores padrao.')

# 2. Ler taxa configurada (do control_state ou fallback)
SAMPLE_RATE_HZ = int(control_state.get('sample_rate', 20))
EFFECTIVE_SAMPLE_RATE_HZ = SAMPLE_RATE_HZ

# Sampling validation metadata (preenchido mais adiante; evita NameError na exportacao)
VALIDATION_METADATA = {
    'status': 'not_computed_yet',
    'expected_hz': float(EFFECTIVE_SAMPLE_RATE_HZ),
    'per_class': {},
}

# --- PARAMETROS DE FILTRO (opcionais) ---
FILTER_SAMPLE_RATE_HZ = None  # ex: 20 para filtrar; None = usar todas
FILTER_COLLECTION_ID = None   # ex: 'col_...' ou ['col_a', 'col_b']

# 7 classes compostas (derivadas de cmd_speed_label + rot_state_label)
FILTER_LABELS = CLASS_ORDER


# 3. Consultar Oracle para obter parametros reais da coleta
def _rows_to_dict(cursor, rows):
    """Converte resultado de cursor Oracle para lista de dicts."""
    cols = [d[0].lower() for d in cursor.description]
    return [dict(zip(cols, row)) for row in rows]

print(f'\n[AUTO] Consultando Oracle para validar parametros...')
try:
    _conn_auto = oracledb.connect(
        user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN_STR
    )
    _cursor = _conn_auto.cursor()

    # Total de amostras por cmd_speed_label
    _cursor.execute("""
        SELECT cmd_speed_label AS fan_state, COUNT(*) AS cnt,
               MIN(ts_epoch) AS ts_min, MAX(ts_epoch) AS ts_max,
               AVG(sample_rate) AS avg_sample_rate
        FROM sensor_data
        WHERE cmd_speed_label IN ('LOW', 'MEDIUM', 'HIGH', 'OFF')
        GROUP BY cmd_speed_label
        ORDER BY cnt DESC
    """)
    _class_stats = _rows_to_dict(_cursor, _cursor.fetchall())

    # Collections disponiveis
    _cursor.execute("""
        SELECT collection_id, COUNT(*) AS cnt,
               MIN(ts_epoch) AS ts_min, MAX(ts_epoch) AS ts_max
        FROM sensor_data
        WHERE cmd_speed_label IN ('LOW', 'MEDIUM', 'HIGH')
        GROUP BY collection_id
        ORDER BY MAX(ts_epoch) DESC
    """)
    _collection_stats = _rows_to_dict(_cursor, _cursor.fetchall())

    # Taxa real de sample_rate gravada no banco
    _cursor.execute("""
        SELECT ROUND(AVG(sample_rate), 2) AS db_avg_rate,
               ROUND(MIN(sample_rate), 2) AS db_min_rate,
               ROUND(MAX(sample_rate), 2) AS db_max_rate
        FROM sensor_data
        WHERE cmd_speed_label IN ('LOW', 'MEDIUM', 'HIGH')
    """)
    _rate_rows = _rows_to_dict(_cursor, _cursor.fetchall())
    _rate_stats = _rate_rows[0] if _rate_rows else {}

    _conn_auto.close()

    # Exibir resumo
    print(f'\n{"="*60}')
    print(f' PARAMETROS AUTO-DETECTADOS DO SISTEMA (Oracle)')
    print(f'{"="*60}')
    print(f'  SAMPLE_RATE_HZ (control_state): {SAMPLE_RATE_HZ} Hz')
    if _rate_stats and _rate_stats.get('db_avg_rate') is not None:
        print(f'  sample_rate no banco (media):   {_rate_stats["db_avg_rate"]} Hz')
        print(f'  sample_rate no banco (min/max):  {_rate_stats["db_min_rate"]} / {_rate_stats["db_max_rate"]} Hz')
        db_rate = float(_rate_stats['db_avg_rate'])
        if abs(db_rate - SAMPLE_RATE_HZ) / max(SAMPLE_RATE_HZ, 1) > 0.15:
            print(f'  *** ALERTA: Taxa no banco ({db_rate} Hz) difere do control_state ({SAMPLE_RATE_HZ} Hz)! ***')

    print(f'\n  Dados por cmd_speed_label no banco:')
    _total_training = 0
    for row in _class_stats:
        state = row['fan_state']
        count = int(row['cnt'])
        duration = (float(row['ts_max']) - float(row['ts_min'])) / 60 if row['ts_max'] and row['ts_min'] else 0
        marker = '  (inativo)' if state == 'OFF' else ''
        print(f'    {state:8s}: {count:6d} amostras, {duration:6.1f} min{marker}')
        if state in ('LOW', 'MEDIUM', 'HIGH'):
            _total_training += count
    print(f'    {"TOTAL":8s}: {_total_training:6d} amostras de treino')

    if _collection_stats:
        print(f'\n  Collections (treino):')
        for row in _collection_stats:
            duration = (float(row['ts_max']) - float(row['ts_min'])) / 60 if row['ts_max'] and row['ts_min'] else 0
            print(f'    {row["collection_id"]}: {int(row["cnt"])} amostras, {duration:.1f} min')

    print(f'{"="*60}')

except Exception as e:
    print(f'[AVISO] Nao foi possivel consultar Oracle: {e}')
    print(f'  Verifique se o servico Oracle esta rodando e as credenciais estao corretas.')
    print(f'  Usando SAMPLE_RATE_HZ = {SAMPLE_RATE_HZ} do control_state.')

# --- PARAMETROS DE FEATURE ENGINEERING ---
WINDOW_SIZE = 100    # pontos por janela
STEP_SIZE = 20       # stride da janela deslizante

# Feature selection
ANOVA_ALPHA = 0.05
CORRELATION_THRESHOLD = 0.85

# Eixos do sensor
SENSOR_AXES = ['accel_x_g', 'accel_y_g', 'accel_z_g', 'gyro_x_dps', 'gyro_y_dps', 'gyro_z_dps']

# Eixos/metricas candidatos do modelo
# Metricas estendidas: basicas + shape + percentis + FFT por banda
DERIVED_AXES = ['vibration_dps', 'accel_mag_g']
FEATURE_AXES = SENSOR_AXES + DERIVED_AXES
FEATURE_METRICS = [
    'std', 'range', 'rms',            # basicas (alinham com JS)
    'skew', 'kurtosis',               # shape
    'p10', 'p25', 'p75', 'p90', 'p95',  # percentis
    'fft_low', 'fft_mid', 'fft_high', # bandas FFT (0-5 Hz, 5-15 Hz, 15-10 Hz)
]
FEATURE_COLUMNS = [f'{ax}_{m}' for ax in FEATURE_AXES for m in FEATURE_METRICS]

# Suavizacao para graficos (apenas visual)
SMOOTH_WINDOW_S = 0.5
SMOOTH_POLYORDER = 3

# Paths
from shared.paths import get_paths
PATHS = get_paths()
TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DATA = str(PATHS.output_data)
OUTPUT_FIGURES = str(PATHS.output_figures)
OUTPUT_METRICS = str(PATHS.output_metrics)
CONFIG_DIR = str(PATHS.config_dir)

def _savgol_smooth(series, sample_rate_hz, window_s=SMOOTH_WINDOW_S, polyorder=SMOOTH_POLYORDER):
    values = np.asarray(series, dtype=float)
    if values.size < 3:
        return values
    window_len = int(round(window_s * sample_rate_hz))
    if window_len < 3:
        window_len = 3
    if window_len % 2 == 0:
        window_len += 1
    if window_len > values.size:
        window_len = values.size if values.size % 2 == 1 else values.size - 1
    if window_len < 3:
        return values
    if polyorder >= window_len:
        polyorder = max(1, window_len - 1)
    try:
        return savgol_filter(values, window_length=window_len, polyorder=polyorder)
    except Exception:
        return values

print(f'\n--- Configuracao Final ---')
print(f'Timestamp:            {TIMESTAMP_STR}')
print(f'Oracle DSN:           {ORACLE_DSN_STR}')
print(f'Sample Rate:          {SAMPLE_RATE_HZ} Hz')
if FILTER_SAMPLE_RATE_HZ is not None:
    print(f'Filtro sample_rate:   {FILTER_SAMPLE_RATE_HZ} Hz')
if FILTER_COLLECTION_ID:
    print(f'Filtro collection_id: {FILTER_COLLECTION_ID}')
if FILTER_LABELS:
    print(f'Filtro labels:        {FILTER_LABELS}')
print(f'Window: {WINDOW_SIZE} pontos = {WINDOW_SIZE/SAMPLE_RATE_HZ:.0f}s, Step: {STEP_SIZE} pontos = {STEP_SIZE/SAMPLE_RATE_HZ:.0f}s')
print(f'Features candidatas por eixo: {len(FEATURE_METRICS)} metricas x {len(FEATURE_AXES)} eixos = {len(FEATURE_COLUMNS)} features')
print(f'Figuras:  {os.path.abspath(OUTPUT_FIGURES)}')
print(f'Metricas: {os.path.abspath(OUTPUT_METRICS)}')

[AUTO] control_state.json carregado: ..\api\control_states\control_state_ESP32_MPU6050_ORACLE.json
  Modo atual:      PAUSE
  Sample rate:     20 Hz
  Collection ID:   col_20260222_112119_20hz

[AUTO] Consultando Oracle para validar parametros...

 PARAMETROS AUTO-DETECTADOS DO SISTEMA (Oracle)
  SAMPLE_RATE_HZ (control_state): 20 Hz

  Dados por cmd_speed_label no banco:
    OFF     :     20 amostras,    0.0 min  (inativo)
    TOTAL   :      0 amostras de treino

--- Configuracao Final ---
Timestamp:            20260222_152508
Oracle DSN:           localhost:1521/xepdb1
Sample Rate:          20 Hz
Filtro labels:        ['LOW_ROT_ON', 'MEDIUM_ROT_ON', 'HIGH_ROT_ON', 'LOW_ROT_OFF', 'MEDIUM_ROT_OFF', 'HIGH_ROT_OFF', 'FAN_OFF']
Window: 100 pontos = 5s, Step: 20 pontos = 1s
Features candidatas por eixo: 13 metricas x 8 eixos = 104 features
Figuras:  c:\Users\Anderson\Downloads\oracle_iot_esp32_MPU6050_project-main\notebooks\output\figures
Metricas: c:\Users\Anderson\Downloads\oracle_iot_es

In [13]:
# =============================================================================
# Celula 2: Carga do banco de dados
# =============================================================================
if 'DATA_SOURCE' not in globals():
    DATA_SOURCE = 'SQL'
if 'CSV_FILE' not in globals():
    CSV_FILE = None

RAW_SOURCE_PATH = None

def _apply_filters_df(df):
    """Aplica filtros no DataFrame. Deriva composite label se possivel."""
    df_filtered = df.copy()
    df_filtered = derive_composite_label(df_filtered)

    if 'fan_state' in df_filtered.columns:
        if FILTER_LABELS:
            df_filtered = df_filtered[df_filtered['fan_state'].isin(FILTER_LABELS)].copy()
        else:
            df_filtered = df_filtered[df_filtered['fan_state'].isin(LEGACY_CLASS_ORDER)].copy()
    if FILTER_COLLECTION_ID and 'collection_id' in df_filtered.columns:
        if isinstance(FILTER_COLLECTION_ID, (list, tuple, set)):
            df_filtered = df_filtered[df_filtered['collection_id'].isin(list(FILTER_COLLECTION_ID))].copy()
        else:
            df_filtered = df_filtered[df_filtered['collection_id'] == FILTER_COLLECTION_ID].copy()
    if FILTER_SAMPLE_RATE_HZ is not None and 'sample_rate' in df_filtered.columns:
        df_filtered = df_filtered[df_filtered['sample_rate'] == FILTER_SAMPLE_RATE_HZ].copy()
    return df_filtered

if DATA_SOURCE == 'CSV':
    data_dir = OUTPUT_DATA if 'OUTPUT_DATA' in globals() else os.path.join('output', 'data')
    if not CSV_FILE:
        raise ValueError('Fonte CSV selecionada, mas nenhum arquivo foi escolhido. Use a Celula 0.')
    csv_path = os.path.join(data_dir, CSV_FILE)
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f'CSV nao encontrado: {csv_path}')
    df_raw = pd.read_csv(csv_path)
    RAW_SOURCE_PATH = csv_path
    df_raw = _apply_filters_df(df_raw)
    print(f'CSV carregado: {csv_path}')
else:
    # --- Carga via Oracle usando load_raw_from_oracle() de shared/data_sources.py ---
    _cid = FILTER_COLLECTION_ID if isinstance(FILTER_COLLECTION_ID, str) else None
    df_raw = load_raw_from_oracle(
        connection_str=ORACLE_CONN_STR,
        collection_id=_cid,
    )
    RAW_SOURCE_PATH = f'Oracle:{ORACLE_DSN_STR}'

    # Derivar label composto e aplicar filtros em Python
    df_raw = _apply_filters_df(df_raw)
    print('Dados carregados do Oracle (com labels compostos).')

if df_raw.empty:
    raise ValueError('Nenhum dado encontrado apos aplicar os filtros.')

# Normalizar tipos
if 'sample_rate' in df_raw.columns:
    df_raw['sample_rate'] = pd.to_numeric(df_raw['sample_rate'], errors='coerce')

# Atualizar taxa efetiva (usa filtro ou taxa unica no banco)
if 'sample_rate' in df_raw.columns and df_raw['sample_rate'].notna().any():
    unique_rates = sorted(df_raw['sample_rate'].dropna().unique().tolist())
    if FILTER_SAMPLE_RATE_HZ is not None:
        EFFECTIVE_SAMPLE_RATE_HZ = float(FILTER_SAMPLE_RATE_HZ)
    elif len(unique_rates) == 1:
        EFFECTIVE_SAMPLE_RATE_HZ = float(unique_rates[0])
    else:
        print(f'[AVISO] Multiplas sample_rate no dataset: {unique_rates}')
        print('[AVISO] Recomenda-se filtrar por sample_rate para consistencia.')
else:
    print('[AVISO] Coluna sample_rate nao encontrada; usando control_state.')

# --- Eixos derivados (para enriquecer pool de features) ---
if 'vibration_dps' not in df_raw.columns and 'vibration' in df_raw.columns:
    df_raw['vibration_dps'] = pd.to_numeric(df_raw['vibration'], errors='coerce')

if 'accel_mag_g' not in df_raw.columns and all(c in df_raw.columns for c in ('accel_x_g', 'accel_y_g', 'accel_z_g')):
    ax = pd.to_numeric(df_raw['accel_x_g'], errors='coerce')
    ay = pd.to_numeric(df_raw['accel_y_g'], errors='coerce')
    az = pd.to_numeric(df_raw['accel_z_g'], errors='coerce')
    df_raw['accel_mag_g'] = np.sqrt(ax**2 + ay**2 + az**2)

# Determinar classes ativas e cores baseado nos dados carregados
ACTIVE_CLASSES = get_active_classes(df_raw)
ACTIVE_COLOR_MAP = get_color_map(df_raw)

print(f'[INFO] Amostras carregadas: {len(df_raw)}')
print(f'[INFO] EFFECTIVE_SAMPLE_RATE_HZ: {EFFECTIVE_SAMPLE_RATE_HZ} Hz')
print(f'[INFO] Classes ativas: {ACTIVE_CLASSES}')
print(f'[INFO] Distribuicao: {df_raw["fan_state"].value_counts().to_dict()}')

CSV carregado: c:\Users\Anderson\Downloads\oracle_iot_esp32_MPU6050_project-main\notebooks\output\data\raw_sensor_data_20260222_120323.csv
[INFO] Amostras carregadas: 49840
[INFO] EFFECTIVE_SAMPLE_RATE_HZ: 20.0 Hz
[INFO] Classes ativas: ['LOW_ROT_ON', 'MEDIUM_ROT_ON', 'HIGH_ROT_ON', 'LOW_ROT_OFF', 'MEDIUM_ROT_OFF', 'HIGH_ROT_OFF', 'FAN_OFF']
[INFO] Distribuicao: {'FAN_OFF': 10240, 'LOW_ROT_ON': 6600, 'MEDIUM_ROT_ON': 6600, 'HIGH_ROT_ON': 6600, 'LOW_ROT_OFF': 6600, 'MEDIUM_ROT_OFF': 6600, 'HIGH_ROT_OFF': 6600}


In [14]:
# =============================================================================
# Celula 3: Salvar dados brutos em CSV (backup do banco) com timestamp legivel
# =============================================================================
# Adicionar timestamp ISO 8601 legivel
df_raw['timestamp_iso'] = pd.to_datetime(df_raw['timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')

# Adicionar informacao de frequencia de amostragem
df_raw['sample_rate_configured_hz'] = EFFECTIVE_SAMPLE_RATE_HZ

# Calcular taxa real de amostragem por classe
ts_col = df_raw['timestamp'].copy()
if ts_col.median() > 1e12:
    ts_col = ts_col / 1000.0

for cls in df_raw['fan_state'].unique():
    mask = df_raw['fan_state'] == cls
    t0 = ts_col[mask].min()
    duration = ts_col[mask].max() - t0
    n_samples = mask.sum()
    real_rate = n_samples / duration if duration > 0 else 0
    df_raw.loc[mask, 'sample_rate_real_hz'] = round(real_rate, 2)

raw_csv_path = os.path.join(OUTPUT_DATA, f'raw_sensor_data_{TIMESTAMP_STR}.csv')
df_raw.to_csv(raw_csv_path, index=False)
print(f'Dados brutos salvos em: {raw_csv_path}')
print(f'Shape: {df_raw.shape}')
print(f'Colunas: {list(df_raw.columns)}')
print(f'Exemplo timestamp_iso: {df_raw["timestamp_iso"].iloc[0]}')
print(f'Taxa configurada: {EFFECTIVE_SAMPLE_RATE_HZ} Hz')
print('Taxa real por classe:')
for cls in ACTIVE_CLASSES:
    if cls in df_raw['fan_state'].unique():
        rate = df_raw.loc[df_raw['fan_state'] == cls, 'sample_rate_real_hz'].iloc[0]
        print(f'  {cls}: {rate} Hz')
    else:
        print(f'  {cls}: n/a')

Dados brutos salvos em: c:\Users\Anderson\Downloads\oracle_iot_esp32_MPU6050_project-main\notebooks\output\data\raw_sensor_data_20260222_152508.csv
Shape: (49840, 26)
Colunas: ['id', 'timestamp', 'temperature', 'vibration', 'accel_x_g', 'accel_y_g', 'accel_z_g', 'gyro_x_dps', 'gyro_y_dps', 'gyro_z_dps', 'sample_rate', 'fan_state', 'collection_id', 'cmd_speed_label', 'rot_state_label', 'use_state_label', 'vib_profile_label', 'label_source', 'transition_marker', 'device_id', 'created_at', 'timestamp_iso', 'vibration_dps', 'accel_mag_g', 'sample_rate_configured_hz', 'sample_rate_real_hz']
Exemplo timestamp_iso: 2026-02-22T14:21:31.154722Z
Taxa configurada: 20.0 Hz
Taxa real por classe:
  LOW_ROT_ON: 20.0 Hz
  MEDIUM_ROT_ON: 20.02 Hz
  HIGH_ROT_ON: 20.01 Hz
  LOW_ROT_OFF: 20.0 Hz
  MEDIUM_ROT_OFF: 20.0 Hz
  HIGH_ROT_OFF: 20.0 Hz
  FAN_OFF: 19.03 Hz


In [15]:
# =============================================================================
# Celula 4: Normalizacao do tempo relativo por classe + taxa real
# =============================================================================
df_raw['timestamp_s'] = df_raw['timestamp'].copy()
# Se timestamp estiver em ms, converter para s
if df_raw['timestamp_s'].median() > 1e12:
    df_raw['timestamp_s'] = df_raw['timestamp_s'] / 1000.0

# Tempo relativo por classe (segundos desde o inicio da coleta daquela classe)
df_raw['relative_time_s'] = 0.0
for cls in df_raw['fan_state'].unique():
    mask = df_raw['fan_state'] == cls
    t0 = df_raw.loc[mask, 'timestamp_s'].min()
    df_raw.loc[mask, 'relative_time_s'] = df_raw.loc[mask, 'timestamp_s'] - t0

# Duracao e taxa real por classe
duration_per_class = df_raw.groupby('fan_state')['relative_time_s'].max()
samples_per_class = df_raw.groupby('fan_state').size()

print('=== Duracao por classe (s) ===')
print(duration_per_class)
print()
print(f'Amostras por classe: {samples_per_class.to_dict()}')
print()
print('=== Taxa real de amostragem (Hz) ===')
for cls in ACTIVE_CLASSES:
    if cls not in samples_per_class or cls not in duration_per_class or duration_per_class[cls] == 0:
        print(f'  {cls}: n/a')
        continue
    real_rate = samples_per_class[cls] / duration_per_class[cls]
    print(f'  {cls}: {real_rate:.2f} Hz (configurada: {EFFECTIVE_SAMPLE_RATE_HZ} Hz)')

# Consolidar metadata de validacao de taxa (com base nos dados carregados)
expected_hz = float(EFFECTIVE_SAMPLE_RATE_HZ) if 'EFFECTIVE_SAMPLE_RATE_HZ' in globals() else None
_sampling_stats = {}
for _cls in ACTIVE_CLASSES:
    if _cls not in samples_per_class or _cls not in duration_per_class or duration_per_class[_cls] == 0:
        continue
    _actual = float(samples_per_class[_cls] / duration_per_class[_cls])
    _dev = abs(_actual - expected_hz) / max(expected_hz, 1) * 100 if expected_hz is not None else None
    _sampling_stats[_cls] = {
        'samples': int(samples_per_class[_cls]),
        'duration_s': float(duration_per_class[_cls]),
        'actual_hz': round(_actual, 2),
        'deviation_pct': round(_dev, 2) if _dev is not None else None,
    }

VALIDATION_METADATA = {
    'status': 'computed_from_loaded_data',
    'expected_hz': expected_hz,
    'per_class': _sampling_stats,
    'note': f'Taxa estimada via timestamps das {len(ACTIVE_CLASSES)} classes ativas.',
}

=== Duracao por classe (s) ===
fan_state
FAN_OFF           538.161458
HIGH_ROT_OFF      329.964651
HIGH_ROT_ON       329.887206
LOW_ROT_OFF       329.990613
LOW_ROT_ON        329.951374
MEDIUM_ROT_OFF    329.932658
MEDIUM_ROT_ON     329.740445
Name: relative_time_s, dtype: float64

Amostras por classe: {'FAN_OFF': 10240, 'HIGH_ROT_OFF': 6600, 'HIGH_ROT_ON': 6600, 'LOW_ROT_OFF': 6600, 'LOW_ROT_ON': 6600, 'MEDIUM_ROT_OFF': 6600, 'MEDIUM_ROT_ON': 6600}

=== Taxa real de amostragem (Hz) ===
  LOW_ROT_ON: 20.00 Hz (configurada: 20.0 Hz)
  MEDIUM_ROT_ON: 20.02 Hz (configurada: 20.0 Hz)
  HIGH_ROT_ON: 20.01 Hz (configurada: 20.0 Hz)
  LOW_ROT_OFF: 20.00 Hz (configurada: 20.0 Hz)
  MEDIUM_ROT_OFF: 20.00 Hz (configurada: 20.0 Hz)
  HIGH_ROT_OFF: 20.00 Hz (configurada: 20.0 Hz)
  FAN_OFF: 19.03 Hz (configurada: 20.0 Hz)


In [16]:
# =============================================================================
# Celula 5: Calculo da janela de tempo comum
# =============================================================================
common_window_s = duration_per_class.min()
print(f'Janela de tempo comum: {common_window_s:.2f} s')
print(f'Isso corresponde a ~{common_window_s/60:.1f} minutos')

Janela de tempo comum: 329.74 s
Isso corresponde a ~5.5 minutos


In [17]:
# =============================================================================
# Celula 6: FILTRO TEMPORAL - recortar TODAS as classes pela janela comum
# ANTES de qualquer analise posterior (graficos E features)
# =============================================================================
df = df_raw[df_raw['relative_time_s'] <= common_window_s].copy()

print(f'Amostras ANTES do filtro: {len(df_raw)}')
print(f'Amostras APOS o filtro:  {len(df)}')
print(f'Amostras por classe (filtrado): {df.groupby("fan_state").size().to_dict()}')
print(f'Duracao por classe (filtrado):')
print(df.groupby('fan_state')['relative_time_s'].max())

Amostras ANTES do filtro: 49840
Amostras APOS o filtro:  45737
Amostras por classe (filtrado): {'FAN_OFF': 6160, 'HIGH_ROT_OFF': 6595, 'HIGH_ROT_ON': 6597, 'LOW_ROT_OFF': 6594, 'LOW_ROT_ON': 6595, 'MEDIUM_ROT_OFF': 6596, 'MEDIUM_ROT_ON': 6600}
Duracao por classe (filtrado):
fan_state
FAN_OFF           308.012632
HIGH_ROT_OFF      329.714651
HIGH_ROT_ON       329.737206
LOW_ROT_OFF       329.690613
LOW_ROT_ON        329.701374
MEDIUM_ROT_OFF    329.732658
MEDIUM_ROT_ON     329.740445
Name: relative_time_s, dtype: float64


In [18]:
# =============================================================================
# Celula 7: Engenharia de Features (janela deslizante - features estendidas)
#
# extract_features_windowed_extended calcula por janela:
#   - Basicas: std, range, rms  (alinhadas com JS, ddof=0)
#   - Shape:   skewness, kurtosis (excess)
#   - Percentis: P10, P25, P75, P90, P95
#   - FFT bands: fft_low (0-5Hz), fft_mid (5-15Hz), fft_high (15-10Hz)
#
# Total: 13 metricas x 8 eixos = 104 features candidatas
# =============================================================================

import importlib
import shared.feature_engineering as _fe_mod
importlib.reload(_fe_mod)
from shared.feature_engineering import extract_features_windowed_extended

# Aplicar sobre os dados FILTRADOS
all_features = []
for cls in ACTIVE_CLASSES:
    df_cls = df[df['fan_state'] == cls].reset_index(drop=True)
    if df_cls.empty:
        print(f'{cls}: 0 janelas (sem dados)')
        continue
    features = extract_features_windowed_extended(
        df_cls,
        cls,
        sensor_axes=FEATURE_AXES,
        window_size=WINDOW_SIZE,
        step_size=STEP_SIZE,
        timestamp_col='timestamp_s',
        sampling_hz=float(EFFECTIVE_SAMPLE_RATE_HZ),
    )
    all_features.extend(features)
    print(f'{cls}: {len(features)} janelas de {len(df_cls)} amostras')

df_features = pd.DataFrame(all_features)

# Colunas de features (numericas, excluindo metadados)
meta_cols = ['fan_state', 'collection_id', 'window_start', 'window_end', 'timestamp_start', 'timestamp_end', 'timestamp_mean']
feature_cols = [c for c in FEATURE_COLUMNS if c in df_features.columns]

print()
print(f'Total de janelas: {len(df_features)}')
print(f'Features por janela: {len(feature_cols)} (de {len(FEATURE_COLUMNS)} candidatas)')
print(f'Distribuicao: {df_features["fan_state"].value_counts().to_dict()}')

LOW_ROT_ON: 325 janelas de 6595 amostras
MEDIUM_ROT_ON: 326 janelas de 6600 amostras
HIGH_ROT_ON: 325 janelas de 6597 amostras
LOW_ROT_OFF: 325 janelas de 6594 amostras
MEDIUM_ROT_OFF: 325 janelas de 6596 amostras
HIGH_ROT_OFF: 325 janelas de 6595 amostras
FAN_OFF: 304 janelas de 6160 amostras

Total de janelas: 2255
Features por janela: 104 (de 104 candidatas)
Distribuicao: {'MEDIUM_ROT_ON': 326, 'LOW_ROT_ON': 325, 'HIGH_ROT_ON': 325, 'LOW_ROT_OFF': 325, 'MEDIUM_ROT_OFF': 325, 'HIGH_ROT_OFF': 325, 'FAN_OFF': 304}


In [19]:
# =============================================================================
# Celula 8: Selecao de features - Cohen's d + correlacao por classe + pairwise + TOP-K
# =============================================================================

# Recarregar modulo para garantir que mudancas no .py sejam refletidas
import importlib
import shared.feature_selection as _fs_mod
importlib.reload(_fs_mod)
from shared.feature_selection import (
    select_features_anova_classwise_corr_pairwise_score_topk,
    select_features_cohens_d_classwise_corr_pairwise_score_topk,
)

# Parametros (ajuste fino aqui)
SELECTION_METHOD = 'cohens_d'  # 'cohens_d' (recomendado) ou 'anova'

# Cohen's d: threshold realista para 7 classes com assinaturas sobrepostas
# min_cohens_d=0.3 => separacao pequena-media (d >= 0.3 em TODOS os 21 pares)
# Anteriormente 2.0 (nunca atingido); 0.3 seleciona features genuinamente discriminativas
MIN_COHENS_D = 0.3
COHENS_SCORE_MODE = 'd_min_all'  # d_min_all => score = min de todos os 21 pares

# Filtros e TOP-K
CORRELATION_THRESHOLD = 0.85
PAIRWISE_MIN_SEPARATION = 1.0
PAIRWISE_MIN_PAIRS = 2
TOP_K_FEATURES = min(16, len(feature_cols))
CORRELATION_MODE = 'classwise_median'
SCORE_FORMULA = COHENS_SCORE_MODE if SELECTION_METHOD == 'cohens_d' else 'min_sep * log(1 + F)'

CLASSES = ACTIVE_CLASSES

# Aviso sobre classes com poucos dados
_class_window_counts = df_features['fan_state'].value_counts()
_small_classes = _class_window_counts[_class_window_counts < 5]
if not _small_classes.empty:
    print(f'[AVISO] Classes com poucas janelas (< 5):')
    for cls, cnt in _small_classes.items():
        print(f'  {cls}: {cnt} janelas')
    print(f'  Estas classes podem nao contribuir para ANOVA/Cohen\'s d.\n')

# 1) Selecao principal
if SELECTION_METHOD == 'anova':
    ANOVA_ALPHA_SELECTION = ANOVA_ALPHA
    selected_features, df_scores, df_anova, df_significant = (
        select_features_anova_classwise_corr_pairwise_score_topk(
            df_features,
            feature_cols,
            class_col='fan_state',
            classes=CLASSES,
            anova_alpha=ANOVA_ALPHA_SELECTION,
            correlation_threshold=CORRELATION_THRESHOLD,
            correlation_mode=CORRELATION_MODE,
            pairwise_min_separation=PAIRWISE_MIN_SEPARATION,
            pairwise_min_pairs=PAIRWISE_MIN_PAIRS,
            top_k=TOP_K_FEATURES,
            verbose=True,
        )
    )

    # Diagnostico: Cohen's d (sem filtros) para ranking/analise
    _, _, df_sep = select_features_cohens_d_classwise_corr_pairwise_score_topk(
        df_features,
        feature_cols,
        class_col='fan_state',
        classes=CLASSES,
        correlation_threshold=1.0,
        correlation_mode=CORRELATION_MODE,
        pairwise_min_separation=0.0,
        pairwise_min_pairs=0,
        min_cohens_d=None,
        score_mode=COHENS_SCORE_MODE,
        top_k=len(feature_cols),
        verbose=False,
    )
else:
    selected_features, df_scores, df_sep = (
        select_features_cohens_d_classwise_corr_pairwise_score_topk(
            df_features,
            feature_cols,
            class_col='fan_state',
            classes=CLASSES,
            correlation_threshold=CORRELATION_THRESHOLD,
            correlation_mode=CORRELATION_MODE,
            pairwise_min_separation=PAIRWISE_MIN_SEPARATION,
            pairwise_min_pairs=PAIRWISE_MIN_PAIRS,
            min_cohens_d=MIN_COHENS_D,
            score_mode=COHENS_SCORE_MODE,
            top_k=TOP_K_FEATURES,
            verbose=True,
        )
    )

    # Diagnostico: ANOVA (sem filtros) para manter a validacao robusta (celula 9)
    _, _, df_anova, df_significant = (
        select_features_anova_classwise_corr_pairwise_score_topk(
            df_features,
            feature_cols,
            class_col='fan_state',
            classes=CLASSES,
            anova_alpha=ANOVA_ALPHA,
            correlation_threshold=1.0,
            correlation_mode=CORRELATION_MODE,
            pairwise_min_separation=0.0,
            pairwise_min_pairs=0,
            top_k=len(feature_cols),
            verbose=False,
        )
    )
    ANOVA_ALPHA_SELECTION = ANOVA_ALPHA

print(f'\n--- Resultado ---')
print(f'Features selecionadas: {len(selected_features)}')
if df_anova is not None and not df_anova.empty:
    print(f'ANOVA significativas: {len(df_significant)} de {len(df_anova)}')
else:
    print(f'ANOVA: sem resultados (possivelmente classes com dados insuficientes)')

Cohen's d ranking (top 15, 21 pares):
              feature  d_min_all
       gyro_x_dps_p90   0.540452
  vibration_dps_range   0.539697
    accel_y_g_fft_low   0.442624
    vibration_dps_p10   0.426935
    accel_z_g_fft_low   0.421161
        accel_y_g_std   0.398666
      accel_mag_g_p95   0.383989
        accel_y_g_rms   0.342552
    vibration_dps_p90   0.330151
        accel_x_g_p90   0.321565
vibration_dps_fft_mid   0.317612
        accel_x_g_std   0.313781
    vibration_dps_p95   0.310075
        accel_x_g_rms   0.301743
      accel_x_g_range   0.268823

AVISO: poucos candidatos com d_min_all >= 0.3 (14). Relaxando criterio para TOP-K por ranking.
Removidas por correlacao > 0.85: 47
Apos correlacao: 57

--- Filtro pairwise (min=1.0, pares>=2) ---
              feature    score  pairs_passed  d_min_all
       gyro_x_dps_p90 0.540452            16   0.540452
  vibration_dps_range 0.539697            19   0.539697
    accel_y_g_fft_low 0.442624            20   0.442624
    vibration

In [20]:
# =============================================================================
# Celula 9: Validacao robusta - Kruskal-Wallis + Mutual Information
#
# Kruskal-Wallis: teste nao-parametrico (nao assume normalidade)
# Mutual Information: captura relacoes nao-lineares entre feature e classe
# =============================================================================

# --- 1. Kruskal-Wallis (nao-parametrico) ---
kw_results = []
kw_skipped = []
for feat in feature_cols:
    groups = [df_features[df_features['fan_state'] == cls][feat].dropna().values
              for cls in ACTIVE_CLASSES]
    groups = [g for g in groups if len(g) > 1]  # remover classes sem dados
    if len(groups) < 2:
        continue
    # Verificar se ha variabilidade suficiente para o teste
    all_vals = np.concatenate(groups)
    if len(np.unique(all_vals)) <= 1:
        kw_skipped.append(feat)
        continue
    try:
        h_stat, p_val = kruskal(*groups)
        kw_results.append({'feature': feat, 'h_statistic': h_stat, 'p_value': p_val})
    except ValueError:
        kw_skipped.append(feat)

if kw_skipped:
    print(f'[INFO] {len(kw_skipped)} features puladas no KW (valores constantes): {kw_skipped}')

df_kw = pd.DataFrame(kw_results)
if not df_kw.empty:
    df_kw = df_kw.sort_values('h_statistic', ascending=False).reset_index(drop=True)
kw_significant = df_kw[df_kw['p_value'] < ANOVA_ALPHA]['feature'].tolist() if not df_kw.empty else []

print(f'=== Kruskal-Wallis ===')
print(f'Features significativas (p < {ANOVA_ALPHA}): {len(kw_significant)} de {len(df_kw)}')

# --- 2. Mutual Information ---
le = LabelEncoder()
y_encoded = le.fit_transform(df_features['fan_state'])
X_features = df_features[feature_cols]

mi_scores = mutual_info_classif(X_features, y_encoded, random_state=42)
df_mi = pd.DataFrame({'feature': feature_cols, 'mi_score': mi_scores}).sort_values('mi_score', ascending=False)

print(f'\n=== Mutual Information ===')
print(f'Top 15 features por MI:')
for _, row in df_mi.head(15).iterrows():
    print(f'  {row["feature"]:40s} MI={row["mi_score"]:.4f}')

# --- 3. Comparacao de rankings ---
df_ranking = pd.DataFrame({'feature': feature_cols})

# ANOVA rank (pode estar vazio se classes com dados insuficientes)
if 'df_anova' in globals() and df_anova is not None and not df_anova.empty:
    anova_rank = {row['feature']: i+1 for i, (_, row) in enumerate(df_anova.iterrows())}
    df_ranking['anova_rank'] = df_ranking['feature'].map(anova_rank)
else:
    df_ranking['anova_rank'] = float('nan')
    print('\n[AVISO] ANOVA nao produziu resultados — ranking ANOVA indisponivel.')

kw_rank = {row['feature']: i+1 for i, (_, row) in enumerate(df_kw.iterrows())} if not df_kw.empty else {}
df_ranking['kw_rank'] = df_ranking['feature'].map(kw_rank)
mi_rank = {row['feature']: i+1 for i, (_, row) in enumerate(df_mi.iterrows())}
df_ranking['mi_rank'] = df_ranking['feature'].map(mi_rank)
df_ranking['avg_rank'] = df_ranking[['anova_rank', 'kw_rank', 'mi_rank']].mean(axis=1)
df_ranking = df_ranking.sort_values('avg_rank')

print(f'\n=== Ranking Consensual (media dos metodos disponiveis) ===')
print(f'Top 20 features:')
for _, row in df_ranking.head(20).iterrows():
    in_selected = '*' if row['feature'] in selected_features else ' '
    anova_str = f'{row["anova_rank"]:3.0f}' if pd.notna(row.get('anova_rank')) else '  -'
    kw_str = f'{row["kw_rank"]:3.0f}' if pd.notna(row.get('kw_rank')) else '  -'
    print(f'  {in_selected} {row["feature"]:40s} ANOVA={anova_str}  KW={kw_str}  MI={row["mi_rank"]:3.0f}  AVG={row["avg_rank"]:.1f}')
print(f'\n  * = presente nas features selecionadas (Top-K)')

# Verificar concordancia
if 'df_significant' in globals() and df_significant is not None and not df_significant.empty:
    anova_set = set(df_significant['feature'].tolist())
else:
    anova_set = set()
kw_set = set(kw_significant)
print(f'\nConcordancia ANOVA vs Kruskal-Wallis:')
print(f'  Ambos significativos: {len(anova_set & kw_set)}')
print(f'  So ANOVA: {len(anova_set - kw_set)}')
print(f'  So KW: {len(kw_set - anova_set)}')

[INFO] 8 features puladas no KW (valores constantes): ['accel_x_g_fft_high', 'accel_y_g_fft_high', 'accel_z_g_fft_high', 'gyro_x_dps_fft_high', 'gyro_y_dps_fft_high', 'gyro_z_dps_fft_high', 'vibration_dps_fft_high', 'accel_mag_g_fft_high']
=== Kruskal-Wallis ===
Features significativas (p < 0.05): 96 de 96

=== Mutual Information ===
Top 15 features por MI:
  vibration_dps_range                      MI=1.5427
  vibration_dps_p95                        MI=1.4836
  accel_x_g_range                          MI=1.4588
  gyro_z_dps_range                         MI=1.4079
  accel_mag_g_range                        MI=1.3921
  accel_z_g_range                          MI=1.3737
  vibration_dps_std                        MI=1.3644
  accel_y_g_range                          MI=1.3599
  gyro_y_dps_range                         MI=1.3494
  vibration_dps_p90                        MI=1.3471
  gyro_z_dps_rms                           MI=1.3431
  vibration_dps_fft_low                    MI=1.3325
  gy

In [21]:
# =============================================================================
# Celula 10: Exportar artefatos
# =============================================================================

from itertools import combinations as _combinations
from shared.traceability import hash_file, save_json, append_registry

# Sampling validation metadata (opcional)
if 'VALIDATION_METADATA' not in globals():
    VALIDATION_METADATA = {
        'status': 'not_available',
        'expected_hz': float(EFFECTIVE_SAMPLE_RATE_HZ) if 'EFFECTIVE_SAMPLE_RATE_HZ' in globals() else None,
        'per_class': {},
        'reason': 'VALIDATION_METADATA nao foi calculada antes da exportacao.',
    }

# Extrair collection_ids usados nos dados
collection_ids_used = df_raw['collection_id'].unique().tolist() if 'collection_id' in df_raw.columns else []

# Garantir collection_id no dataframe de features (para GroupKFold e rastreabilidade)
if 'collection_id' in df_raw.columns and 'collection_id' not in df_features.columns:
    # Mapear collection_id mais frequente por classe
    col_id_per_class = df_raw.groupby('fan_state')['collection_id'].agg(lambda x: x.mode().iloc[0] if len(x) > 0 else 'unknown').to_dict()
    df_features['collection_id'] = df_features['fan_state'].map(col_id_per_class)

# 1. CSV de features extraidas (com timestamp e collection_id)
features_csv_path = os.path.join(OUTPUT_DATA, f'features_extracted_{TIMESTAMP_STR}.csv')
df_features.to_csv(features_csv_path, index=False)
print(f'Features CSV salvo: {features_csv_path}')

# Copia "latest" para o notebook 03
latest_features_path = os.path.join(OUTPUT_DATA, 'features_latest.csv')
df_features.to_csv(latest_features_path, index=False)
print(f'Features latest CSV: {latest_features_path}')

# Hash do dataset de features
features_csv_hash = hash_file(features_csv_path)

# 2. Summary JSON
summary = {
    'timestamp': TIMESTAMP_STR,
    'raw_samples': int(len(df_raw)),
    'filtered_samples': int(len(df)),
    'common_window_s': float(common_window_s),
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'total_windows': int(len(df_features)),
    'feature_count': len(feature_cols),
    'selected_feature_count': len(selected_features),
    'class_distribution_raw': {k: int(v) for k, v in df_raw['fan_state'].value_counts().to_dict().items()},
    'class_distribution_filtered': {k: int(v) for k, v in df['fan_state'].value_counts().to_dict().items()},
    'class_distribution_windows': {k: int(v) for k, v in df_features['fan_state'].value_counts().to_dict().items()},
    'anova_alpha': ANOVA_ALPHA,
    'correlation_threshold': CORRELATION_THRESHOLD,
    'correlation_mode': CORRELATION_MODE,
    'pairwise_min_separation': PAIRWISE_MIN_SEPARATION,
    'pairwise_min_pairs': PAIRWISE_MIN_PAIRS,
    'top_k': TOP_K_FEATURES,
    'score_formula': SCORE_FORMULA,
    'selection_method': (SELECTION_METHOD if 'SELECTION_METHOD' in globals() else None),
    'min_cohens_d': (MIN_COHENS_D if 'MIN_COHENS_D' in globals() else None),
    'cohens_score_mode': (COHENS_SCORE_MODE if 'COHENS_SCORE_MODE' in globals() else None),
    'selected_features': selected_features,
    'sampling_validation': (VALIDATION_METADATA if 'VALIDATION_METADATA' in globals() else None),
    'collection_ids': collection_ids_used,
    'features_csv_hash': features_csv_hash,
    'n_classes': len(ACTIVE_CLASSES),
    'classes': ACTIVE_CLASSES,
}

summary_path = os.path.join(OUTPUT_METRICS, 'analise_exploratoria_summary.json')
save_json(summary_path, summary)
print(f'Summary salvo: {summary_path}')

# 3. Feature config JSON (para frontend e notebook 03)
# --- Versionamento automatico: incrementa minor version a cada execucao ---
config_path = os.path.join(CONFIG_DIR, 'feature_config.json')
previous_version = '0.0'
previous_history = []
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        old_config = json.load(f)
    previous_version = old_config.get('version', '0.0')
    previous_history = old_config.get('version_history', [])

# Incrementar versao: major.minor -> major.(minor+1)
try:
    parts = previous_version.split('.')
    new_version = f'{parts[0]}.{int(parts[1]) + 1}' if len(parts) >= 2 else f'{int(parts[0]) + 1}.0'
except (ValueError, IndexError):
    new_version = '1.0'

# IDs de rastreabilidade
EDA_ID = f'eda_{TIMESTAMP_STR}'
FE_ID = f'fe_{TIMESTAMP_STR}'

# Metodo de selecao (para rastreabilidade)
if 'SELECTION_METHOD' not in globals():
    SELECTION_METHOD = 'anova'
METHOD_ID = (
    'cohens_d_min_adjacent_classwise_corr_pairwise_score_topk'
    if SELECTION_METHOD == 'cohens_d'
    else 'anova_f_test_with_classwise_correlation_pairwise_2of3_score_topk'
)

# Construir cohens_d_ranking dinamicamente (todos os pares presentes no df_sep)
_d_pair_cols = [c for c in df_sep.columns if c.startswith('d_') and c not in ('d_min_adjacent', 'd_min_all')]
_cohens_ranking = []
for _, row in df_sep.iterrows():
    if row.get('feature') not in selected_features:
        continue
    entry = {
        'feature': row['feature'],
        'd_min_all': float(row.get('d_min_all', 0.0)),
    }
    for col in _d_pair_cols:
        entry[col] = float(row.get(col, 0.0))
    _cohens_ranking.append(entry)

feature_config = {
    'version': new_version,
    'eda_id': EDA_ID,
    'fe_id': FE_ID,
    'generated_by': '02_Feature_Engineering.ipynb',
    'generated_at': TIMESTAMP_STR,
    'sample_rate_hz': EFFECTIVE_SAMPLE_RATE_HZ,
    'sampling_validation': (VALIDATION_METADATA if 'VALIDATION_METADATA' in globals() else None),
    'collection_ids': collection_ids_used,
    'features_csv_hash': features_csv_hash,
    'selection_criteria': {
        'anova_alpha': ANOVA_ALPHA,
        'correlation_threshold': CORRELATION_THRESHOLD,
        'correlation_mode': CORRELATION_MODE,
        'pairwise_min_separation': PAIRWISE_MIN_SEPARATION,
        'pairwise_min_pairs': PAIRWISE_MIN_PAIRS,
        'score_formula': SCORE_FORMULA,
        'top_k': TOP_K_FEATURES,
        'method': METHOD_ID,
        'selection_method': SELECTION_METHOD,
        'min_cohens_d': (MIN_COHENS_D if 'MIN_COHENS_D' in globals() else None),
        'cohens_score_mode': (COHENS_SCORE_MODE if 'COHENS_SCORE_MODE' in globals() else None),
    },
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'selected_features': selected_features,
    'all_features': feature_cols,
    'feature_count': len(selected_features),
    'total_feature_count': len(feature_cols),
    'n_classes': len(ACTIVE_CLASSES),
    'classes': ACTIVE_CLASSES,
    'class_distribution': {k: int(v) for k, v in df_features['fan_state'].value_counts().to_dict().items()},
    'sensor_axes': SENSOR_AXES,
    'derived_axes': (DERIVED_AXES if 'DERIVED_AXES' in globals() else []),
    'feature_axes': FEATURE_AXES,
    'feature_metrics': FEATURE_METRICS,
    'cohens_d_ranking': _cohens_ranking,
    'anova_ranking': [
        {
            'feature': row['feature'],
            'f_statistic': float(row['f_statistic']),
            'p_value': float(row['p_value']),
        }
        for _, row in (df_significant.iterrows() if 'df_significant' in globals() and df_significant is not None and not df_significant.empty else iter([]))
        if row['feature'] in selected_features
    ],
    'version_history': previous_history + [
        {
            'version': new_version,
            'eda_id': EDA_ID,
            'fe_id': FE_ID,
            'date': TIMESTAMP_STR,
            'features_count': len(selected_features),
            'n_classes': len(ACTIVE_CLASSES),
            'sample_rate_hz': EFFECTIVE_SAMPLE_RATE_HZ,
            'method': METHOD_ID,
        }
    ],
}

save_json(config_path, feature_config)

print()
print(f'Feature config salvo: {config_path}')
print(f'  Versao: {new_version} (anterior: {previous_version})')
print(f'  EDA ID: {EDA_ID}')
print(f'  FE ID:  {FE_ID}')
print(f'  Features: {len(selected_features)}')
print(f'  Classes: {len(ACTIVE_CLASSES)} ({", ".join(ACTIVE_CLASSES)})')
print(f'  Sample rate: {EFFECTIVE_SAMPLE_RATE_HZ} Hz')
print(f'  Sampling validation: {VALIDATION_METADATA.get("status", "N/A")}')
print(f'  Pairwise separation min (>= 2/3 pares): {PAIRWISE_MIN_SEPARATION}')
print(f'  Collection IDs: {collection_ids_used}')

# 4. Feature engineering run config
fe_run = {
    'generated_at': TIMESTAMP_STR,
    'eda_id': EDA_ID,
    'fe_id': FE_ID,
    'feature_config_version': new_version,
    'raw_csv_path': raw_csv_path if 'raw_csv_path' in globals() else None,
    'features_csv_path': features_csv_path,
    'features_latest_path': latest_features_path,
    'features_csv_hash': features_csv_hash,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'sample_rate_hz': EFFECTIVE_SAMPLE_RATE_HZ,
    'selected_features': selected_features,
    'selection_criteria': feature_config['selection_criteria'],
}

fe_run_path = os.path.join(OUTPUT_METRICS, 'feature_engineering_run.json')
save_json(fe_run_path, fe_run)
print(f'FE run config salvo: {fe_run_path}')

# --- Salvar configuracao resumida da execucao para notebook 03 (compat) ---
eda_run_config = {
    'generated_at': TIMESTAMP_STR,
    'eda_id': EDA_ID,
    'fe_id': FE_ID,
    'feature_config_version': new_version,
    'raw_csv_path': raw_csv_path if 'raw_csv_path' in globals() else None,
    'features_csv_path': features_csv_path,
    'features_latest_path': latest_features_path,
    'features_csv_hash': features_csv_hash,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'sample_rate_hz': EFFECTIVE_SAMPLE_RATE_HZ,
    'selected_features': selected_features,
}

eda_config_path = os.path.join(OUTPUT_METRICS, 'eda_run_config.json')
save_json(eda_config_path, eda_run_config)
print(f'EDA run config salvo: {eda_config_path}')

# 5. Pipeline registry
registry_path = os.path.join(OUTPUT_METRICS, 'pipeline_registry.json')
append_registry(registry_path, {
    'type': 'feature_engineering',
    'timestamp': TIMESTAMP_STR,
    'eda_id': EDA_ID,
    'fe_id': FE_ID,
    'feature_config_version': new_version,
    'features_csv_path': features_csv_path,
    'features_csv_hash': features_csv_hash,
    'selected_features_count': len(selected_features),
    'n_classes': len(ACTIVE_CLASSES),
    'selection_method': feature_config['selection_criteria']['method'],
    'collection_ids': collection_ids_used,
})

print()
print('=== CONCLUIDO ===')
print('Proximo passo: executar 03_Model_Training_Evaluation.ipynb')
print(f'O notebook 03 usara: {latest_features_path}')
print(f'O modelo gerado tera rastreabilidade para EDA ID: {EDA_ID}')

Features CSV salvo: c:\Users\Anderson\Downloads\oracle_iot_esp32_MPU6050_project-main\notebooks\output\data\features_extracted_20260222_152508.csv
Features latest CSV: c:\Users\Anderson\Downloads\oracle_iot_esp32_MPU6050_project-main\notebooks\output\data\features_latest.csv
Summary salvo: c:\Users\Anderson\Downloads\oracle_iot_esp32_MPU6050_project-main\notebooks\output\metrics\analise_exploratoria_summary.json

Feature config salvo: c:\Users\Anderson\Downloads\oracle_iot_esp32_MPU6050_project-main\config\feature_config.json
  Versao: 5.16 (anterior: 5.15)
  EDA ID: eda_20260222_152508
  FE ID:  fe_20260222_152508
  Features: 16
  Classes: 7 (LOW_ROT_ON, MEDIUM_ROT_ON, HIGH_ROT_ON, LOW_ROT_OFF, MEDIUM_ROT_OFF, HIGH_ROT_OFF, FAN_OFF)
  Sample rate: 20.0 Hz
  Sampling validation: computed_from_loaded_data
  Pairwise separation min (>= 2/3 pares): 1.0
  Collection IDs: ['col_20260222_112119_20hz']
FE run config salvo: c:\Users\Anderson\Downloads\oracle_iot_esp32_MPU6050_project-main\note